In [8]:
import sys

import pandas as pd
import os

# Adiciona a pasta pai ao caminho de busca do Python
sys.path.append(os.path.abspath('..'))

pd.options.display.float_format = '{:.2f}'.format

# Carregar os arquivos de 2014
dre = pd.read_csv("../DRE_tratado/dfp_dre_2014_filtrado.csv",              sep=";", encoding="utf-8")
bpa = pd.read_csv("../Balanco_ativo_tratado/dfp_bp_2014_filtrado.csv",     sep=";", encoding="utf-8")
bpp = pd.read_csv("../Balanco_passivo_tratado/dfp_bpp_2014_filtrado.csv",  sep=";", encoding="utf-8")

# Filtrar setor petróleo
dre_pet = dre[dre['setor'] == 'Petroleo']
bpa_pet = bpa[bpa['setor'] == 'Petroleo']
bpp_pet = bpp[bpp['setor'] == 'Petroleo']

print("Empresas na DRE:", dre_pet['empresa'].unique())
print("Empresas no BPA:", bpa_pet['empresa'].unique())
print("Empresas no BPP:", bpp_pet['empresa'].unique())

Empresas na DRE: <StringArray>
['PRIO S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 2, dtype: str
Empresas no BPA: <StringArray>
['PRIO S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 2, dtype: str
Empresas no BPP: <StringArray>
['PRIO S.A.', 'PETROLEO BRASILEIRO S.A. PETROBRAS']
Length: 2, dtype: str


In [9]:
# Ver todas as contas das duas empresas
print("=== PETROBRAS — DRE ===")
pet = dre_pet[dre_pet['empresa'] == 'PETROLEO BRASILEIRO S.A. PETROBRAS']
print(pet[['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string(index=False))

print("\n=== PRIO — DRE ===")
prio = dre_pet[dre_pet['empresa'] == 'PRIO S.A.']
print(prio[['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string(index=False))

=== PETROBRAS — DRE ===
codigo_conta                                                 descricao_conta         valor
        3.01                          Receita de Venda de Bens e/ou Serviços  337260000.00
        3.02                           Custo dos Bens e/ou Serviços Vendidos -256823000.00
        3.03                                                 Resultado Bruto   80437000.00
        3.04                                  Despesas/Receitas Operacionais -102353000.00
     3.04.01                                             Despesas com Vendas  -15974000.00
     3.04.02                               Despesas Gerais e Administrativas  -11223000.00
     3.04.03                      Perdas pela Não Recuperabilidade de Ativos          0.00
     3.04.04                                    Outras Receitas Operacionais          0.00
     3.04.05                                    Outras Despesas Operacionais  -75607000.00
  3.04.05.01                                                     T

In [10]:
print("=== PETROBRAS — BPP ===")
pet_bpp = bpp_pet[bpp_pet['empresa'] == 'PETROLEO BRASILEIRO S.A. PETROBRAS']
print(pet_bpp[['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string(index=False))

print("\n=== PRIO — BPP ===")
prio_bpp = bpp_pet[bpp_pet['empresa'] == 'PRIO S.A.']
print(prio_bpp[['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string(index=False))

print("\n=== CAIXA — BPA ===")
print(bpa_pet[bpa_pet['codigo_conta'] == '1.01.01'][['empresa', 'codigo_conta', 'descricao_conta', 'valor']].to_string(index=False))

=== PETROBRAS — BPP ===
 codigo_conta                                              descricao_conta        valor
            2                                                Passivo Total 793375000.00
         2.01                                           Passivo Circulante  82659000.00
      2.01.01                            Obrigações Sociais e Trabalhistas   5489000.00
   2.01.01.01                                           Obrigações Sociais         0.00
   2.01.01.02                                      Obrigações Trabalhistas         0.00
      2.01.02                                                 Fornecedores  25924000.00
   2.01.02.01                                       Fornecedores Nacionais         0.00
   2.01.02.02                                    Fornecedores Estrangeiros         0.00
      2.01.03                                           Obrigações Fiscais    657000.00
   2.01.03.01                                  Obrigações Fiscais Federais    657000.00
2.01.03.

In [ ]:
# ============================================================
# EXTRAÇÃO E CÁLCULO — PETRÓLEO 2014
# ============================================================

ll        = dre_pet[dre_pet['codigo_conta'] == '3.11.01'][['empresa', 'valor']].rename(columns={'valor': 'lucro_liquido'})
receita   = dre_pet[dre_pet['codigo_conta'] == '3.01'][['empresa', 'valor']].rename(columns={'valor': 'receita'})
res_bruto = dre_pet[dre_pet['codigo_conta'] == '3.03'][['empresa', 'valor']].rename(columns={'valor': 'resultado_bruto'})
ebit      = dre_pet[dre_pet['codigo_conta'] == '3.05'][['empresa', 'valor']].rename(columns={'valor': 'ebit'})
desp_fin  = dre_pet[dre_pet['codigo_conta'] == '3.06.02'][['empresa', 'valor']].rename(columns={'valor': 'desp_financeiras'})
pl        = bpp_pet[bpp_pet['codigo_conta'] == '2.03'][['empresa', 'valor']].rename(columns={'valor': 'patrimonio_liquido'})
div_cp    = bpp_pet[bpp_pet['codigo_conta'] == '2.01.04'][['empresa', 'valor']].rename(columns={'valor': 'div_circulante'})
div_lp    = bpp_pet[bpp_pet['codigo_conta'] == '2.02.01'][['empresa', 'valor']].rename(columns={'valor': 'div_nao_circulante'})
caixa     = bpa_pet[bpa_pet['codigo_conta'] == '1.01.01'][['empresa', 'valor']].rename(columns={'valor': 'caixa'})

# Verificar cobertura
print("=== COBERTURA ===")
for nome, df in [('lucro_liquido', ll), ('receita', receita), ('resultado_bruto', res_bruto),
                 ('ebit', ebit), ('desp_financeiras', desp_fin), ('pl', pl),
                 ('div_cp', div_cp), ('div_lp', div_lp), ('caixa', caixa)]:
    faltando = set(dre_pet['empresa'].unique()) - set(df['empresa'].unique())
    if faltando:
        print(f"❌ {nome}: faltando {faltando}")
    else:
        print(f"✅ {nome}: ambas as empresas")

# Montar e calcular
kpis = ll.merge(receita,   on='empresa')
kpis = kpis.merge(res_bruto, on='empresa')
kpis = kpis.merge(ebit,      on='empresa')
kpis = kpis.merge(desp_fin,  on='empresa')
kpis = kpis.merge(pl,        on='empresa')
kpis = kpis.merge(div_cp,    on='empresa')
kpis = kpis.merge(div_lp,    on='empresa')
kpis = kpis.merge(caixa,     on='empresa')

kpis['roe']            = kpis['lucro_liquido'] / kpis['patrimonio_liquido']
kpis['margem_liquida'] = kpis['lucro_liquido'] / kpis['receita']
kpis['margem_bruta']   = kpis['resultado_bruto'] / kpis['receita']
kpis['divida_bruta']   = kpis['div_circulante'] + kpis['div_nao_circulante']
kpis['divida_liquida'] = kpis['divida_bruta'] - kpis['caixa']
kpis['ebitda']         = kpis.apply(
    lambda r: r['ebit'] if r['ebit'] > 0 else None, axis=1
)
kpis['margem_ebitda']  = kpis.apply(
    lambda r: r['ebitda'] / r['receita'] if r['ebitda'] is not None else None, axis=1
)
kpis['alavancagem']    = kpis.apply(
    lambda r: r['divida_liquida'] / r['ebitda'] if r['ebitda'] is not None and r['ebitda'] > 0 else None, axis=1
)
kpis['cobertura_juros'] = kpis.apply(
    lambda r: r['ebit'] / abs(r['desp_financeiras']) if r['desp_financeiras'] != 0 else None, axis=1
)

from empresas_selecionadas import empresas_selecionadas
kpis['setor'] = kpis['empresa'].map(lambda x: empresas_selecionadas[x][0]) # type: ignore
kpis['tipo']  = kpis['empresa'].map(lambda x: empresas_selecionadas[x][1]) # type: ignore

cols = ['empresa', 'tipo', 'roe', 'margem_liquida', 'margem_bruta',
        'margem_ebitda', 'alavancagem', 'cobertura_juros']
print()
print(kpis[cols].to_string(index=False))

=== COBERTURA ===
✅ lucro_liquido: ambas as empresas
✅ receita: ambas as empresas
✅ resultado_bruto: ambas as empresas
✅ ebit: ambas as empresas
✅ desp_financeiras: ambas as empresas
✅ pl: ambas as empresas
✅ div_cp: ambas as empresas
✅ div_lp: ambas as empresas
✅ caixa: ambas as empresas

                           empresa    tipo   roe  margem_liquida  margem_bruta margem_ebitda alavancagem  cobertura_juros
                         PRIO S.A. Privado -1.61           -2.06          0.04          None        None           -16.29
PETROLEO BRASILEIRO S.A. PETROBRAS Estatal -0.07           -0.06          0.24          None        None            -2.37


In [12]:
# Buscar depreciação nas duas empresas
print("=== DEPRECIAÇÃO — PETROBRAS ===")
print(dre_pet[
    (dre_pet['empresa'] == 'PETROLEO BRASILEIRO S.A. PETROBRAS') &
    (dre_pet['descricao_conta'].str.contains('depreci|amortiz', case=False, na=False))
][['codigo_conta', 'descricao_conta', 'valor']].to_string(index=False))

print("\n=== DEPRECIAÇÃO — PRIO ===")
print(dre_pet[
    (dre_pet['empresa'] == 'PRIO S.A.') &
    (dre_pet['descricao_conta'].str.contains('depreci|amortiz', case=False, na=False))
][['codigo_conta', 'descricao_conta', 'valor']].to_string(index=False))

=== DEPRECIAÇÃO — PETROBRAS ===
Empty DataFrame
Columns: [codigo_conta, descricao_conta, valor]
Index: []

=== DEPRECIAÇÃO — PRIO ===
codigo_conta        descricao_conta     valor
  3.04.02.06 Despesa de Depreciação -10090.00


In [13]:
# Para Petróleo — EBITDA com depreciação da Prio quando disponível
depr = dre_pet[dre_pet['codigo_conta'] == '3.04.02.06'][['empresa', 'valor']].rename(columns={'valor': 'depreciacao'})
kpis = kpis.merge(depr, on='empresa', how='left')
kpis['depreciacao'] = kpis['depreciacao'].fillna(0)

# EBITDA = EBIT + Depreciação (abs)
kpis['ebitda'] = kpis['ebit'] + kpis['depreciacao'].abs()

# Só calcular métricas quando EBITDA > 0
kpis['margem_ebitda']   = kpis.apply(
    lambda r: r['ebitda'] / r['receita'] if r['ebitda'] > 0 else None, axis=1
)
kpis['alavancagem']     = kpis.apply(
    lambda r: r['divida_liquida'] / r['ebitda'] if r['ebitda'] > 0 else None, axis=1
)
kpis['cobertura_juros'] = kpis.apply(
    lambda r: r['ebit'] / abs(r['desp_financeiras']) if r['desp_financeiras'] != 0 else None, axis=1
)

cols = ['empresa', 'tipo', 'roe', 'margem_liquida', 'margem_bruta',
        'margem_ebitda', 'alavancagem', 'cobertura_juros']
print(kpis[cols].to_string(index=False))

                           empresa    tipo   roe  margem_liquida  margem_bruta margem_ebitda alavancagem  cobertura_juros
                         PRIO S.A. Privado -1.61           -2.06          0.04          None        None           -16.29
PETROLEO BRASILEIRO S.A. PETROBRAS Estatal -0.07           -0.06          0.24          None        None            -2.37


In [14]:
import pandas as pd

for ano in range(2014, 2024):
    dre = pd.read_csv(f"../DRE_bruto/dfp_dre_{ano}.csv", sep=";", encoding="latin1")
    
    termos = ['BRAVA', 'PETRORECONCAVO', 'RECÔNCAVO', 'RECONCAVO', 'ENAUTA', '3R PETROLEUM']
    
    matches = []
    for termo in termos:
        found = dre[dre['DENOM_CIA'].str.contains(termo, case=False, na=False)]['DENOM_CIA'].unique()
        matches.extend(found)
    
    if matches:
        print(f"{ano}: {list(set(matches))}")
    else:
        print(f"{ano}: nenhuma encontrada")

2014: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2015: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2016: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2017: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2018: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2019: ['3R PETROLEUM Ã\x93LEO E GÃ\x81S S.A.', 'ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2020: ['BRAVA ENERGIA S.A.', 'ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2021: ['BRAVA ENERGIA S.A.', 'ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2022: ['BRAVA ENERGIA S.A.', 'ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2023: ['BRAVA ENERGIA S.A.', 'ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']


In [15]:
for ano in [2021, 2022, 2023]:
    dre = pd.read_csv(f"../DRE_bruto/dfp_dre_{ano}.csv", sep=";", encoding="latin1")
    matches = dre[dre['DENOM_CIA'].str.contains('PETRO|PETR', case=False, na=False)]['DENOM_CIA'].unique()
    print(f"\n{ano}:")
    for m in matches:
        print(f"  {m}")


2021:
  PETRORECÃNCAVO S.A.
  PETROLEO BRASILEIRO S.A. PETROBRAS
  REFINARIA DE PETROLEOS MANGUINHOS S.A.

2022:
  PETRORECÃNCAVO S.A.
  PETROLEO BRASILEIRO S.A. PETROBRAS
  REFINARIA DE PETROLEOS MANGUINHOS S.A.

2023:
  PETRORECÃNCAVO S.A.
  PETROLEO BRASILEIRO S.A. PETROBRAS
  REFINARIA DE PETROLEOS MANGUINHOS S.A.


In [16]:
import pandas as pd

# Verificar nomes corretos com encoding latin1
for ano in [2019, 2020, 2021]:
    dre = pd.read_csv(
        f"../DRE_bruto/dfp_dre_{ano}.csv",
        sep=";",
        encoding="latin1"  # encoding original dos arquivos CVM
    )
    
    # Buscar todas as empresas de petróleo/gás
    termos = ['BRAVA', 'ENAUTA', '3R PETROLEUM', 'PETRORECONCAVO', 'PETROREC']
    for termo in termos:
        found = dre[
            dre['DENOM_CIA'].str.contains(termo, case=False, na=False)
        ]['DENOM_CIA'].unique()
        if len(found) > 0:
            print(f"{ano} | {termo}: {list(found)}")

2019 | ENAUTA: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2019 | 3R PETROLEUM: ['3R PETROLEUM Ã\x93LEO E GÃ\x81S S.A.']
2020 | BRAVA: ['BRAVA ENERGIA S.A.']
2020 | ENAUTA: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2020 | PETROREC: ['PETRORECÃ\x94NCAVO S.A.']
2021 | BRAVA: ['BRAVA ENERGIA S.A.']
2021 | ENAUTA: ['ENAUTA PARTICIPAÃ\x87Ã\x95ES S.A.']
2021 | PETROREC: ['PETRORECÃ\x94NCAVO S.A.']


In [17]:
import pandas as pd

# Ler com encoding latin1 e decodificar corretamente
# para ver os nomes sem caracteres quebrados
dre_2021 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2021.csv",
    sep=";",
    encoding="latin1"  # encoding original CVM
)

# Filtrar empresas de petróleo e exibir nome limpo
termos = ['BRAVA', 'ENAUTA', '3R PETROLEUM', 'PETROREC']
for termo in termos:
    found = dre_2021[
        dre_2021['DENOM_CIA'].str.contains(termo, case=False, na=False)
    ]['DENOM_CIA'].unique()
    for nome in found:
        # Tentar decodificar o nome corretamente
        try:
            nome_limpo = nome.encode('latin1').decode('utf-8')
        except:
            nome_limpo = nome
        print(f"Original: {nome}")
        print(f"Limpo:    {nome_limpo}")
        print()

Original: BRAVA ENERGIA S.A.
Limpo:    BRAVA ENERGIA S.A.

Original: ENAUTA PARTICIPAÃÃES S.A.
Limpo:    ENAUTA PARTICIPAÇÕES S.A.

Original: PETRORECÃNCAVO S.A.
Limpo:    PETRORECÔNCAVO S.A.



In [18]:
# Confirmar nome correto do 3R Petroleum
dre_2019 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2019.csv",
    sep=";",
    encoding="latin1"  # encoding original CVM
)

# Buscar 3R e decodificar nome
found = dre_2019[
    dre_2019['DENOM_CIA'].str.contains('3R', case=False, na=False)
]['DENOM_CIA'].unique()

for nome in found:
    try:
        # Converter de latin1 para utf-8
        nome_limpo = nome.encode('latin1').decode('utf-8')
    except:
        nome_limpo = nome
    print(f"Original: {nome}")
    print(f"Limpo:    {nome_limpo}")

Original: 3R PETROLEUM ÃLEO E GÃS S.A.
Limpo:    3R PETROLEUM ÓLEO E GÁS S.A.


In [19]:
import pandas as pd

# Carregar DRE de 2014 já filtrada para verificar estrutura
dre_2014 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2014.csv",
    sep=";",
    encoding="latin1"
)

# Filtrar BRB e Banestes
bancos = ['BRB BANCO DE BRASILIA S.A.', 'BANESTES S.A. - BCO EST ESPIRITO SANTO']

for banco in bancos:
    print(f"\n=== {banco} — CONTAS DRE 2014 ===")
    df = dre_2014[dre_2014['DENOM_CIA'] == banco]
    
    # Filtrar só exercício atual e versão mais recente
    df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']
    df = df.sort_values('VERSAO', ascending=False).drop_duplicates(subset=['CD_CONTA'])
    
    # Mostrar contas relevantes para os KPIs
    contas_relevantes = ['3.01', '3.02', '3.03', '3.04.02', '3.04.03', 
                         '3.04.05', '3.09.01', '3.11.01']
    print(df[df['CD_CONTA'].isin(contas_relevantes)][
        ['CD_CONTA', 'DS_CONTA', 'VL_CONTA']
    ].sort_values('CD_CONTA').to_string(index=False))


=== BRB BANCO DE BRASILIA S.A. — CONTAS DRE 2014 ===
Empty DataFrame
Columns: [CD_CONTA, DS_CONTA, VL_CONTA]
Index: []

=== BANESTES S.A. - BCO EST ESPIRITO SANTO — CONTAS DRE 2014 ===
Empty DataFrame
Columns: [CD_CONTA, DS_CONTA, VL_CONTA]
Index: []


In [20]:
import pandas as pd

# Carregar DRE bruta de 2014
dre_2014 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2014.csv",
    sep=";",
    encoding="latin1"
)

# Verificar como aparece o campo ORDEM_EXERC para o BRB
brb = dre_2014[dre_2014['DENOM_CIA'] == 'BRB BANCO DE BRASILIA S.A.']

print("=== VALORES ÚNICOS DE ORDEM_EXERC ===")
print(brb['ORDEM_EXERC'].unique())

print("\n=== PRIMEIRAS LINHAS DO BRB ===")
print(brb[['CD_CONTA', 'DS_CONTA', 'ORDEM_EXERC', 'VL_CONTA']].head(10).to_string(index=False))

=== VALORES ÚNICOS DE ORDEM_EXERC ===
<StringArray>
['PENÃLTIMO', 'ÃLTIMO']
Length: 2, dtype: str

=== PRIMEIRAS LINHAS DO BRB ===
CD_CONTA                                   DS_CONTA ORDEM_EXERC    VL_CONTA
    3.01     Receitas da IntermediaÃ§Ã£o Financeira  PENÃLTIMO  1910786.00
    3.01     Receitas da IntermediaÃ§Ã£o Financeira     ÃLTIMO  2181587.00
    3.02     Despesas da IntermediaÃ§Ã£o Financeira  PENÃLTIMO  -544224.00
    3.02     Despesas da IntermediaÃ§Ã£o Financeira     ÃLTIMO  -808074.00
    3.03 Resultado Bruto IntermediaÃ§Ã£o Financeira  PENÃLTIMO  1366562.00
    3.03 Resultado Bruto IntermediaÃ§Ã£o Financeira     ÃLTIMO  1373513.00
    3.04      Outras Despesas/Receitas Operacionais  PENÃLTIMO -1047714.00
    3.04      Outras Despesas/Receitas Operacionais     ÃLTIMO -1050881.00
 3.04.01       Receitas de PrestaÃ§Ã£o de ServiÃ§os  PENÃLTIMO   416469.00
 3.04.01       Receitas de PrestaÃ§Ã£o de ServiÃ§os     ÃLTIMO   484934.00


In [21]:
import pandas as pd

# Carregar DRE bruta de 2014
dre_2014 = pd.read_csv(
    "../DRE_bruto/dfp_dre_2014.csv",
    sep=";",
    encoding="latin1"
)

# Filtrar BRB e Banestes
bancos = ['BRB BANCO DE BRASILIA S.A.', 'BANESTES S.A. - BCO EST ESPIRITO SANTO']

for banco in bancos:
    print(f"\n=== {banco} — CONTAS DRE 2014 ===")
    df = dre_2014[dre_2014['DENOM_CIA'] == banco].copy()
    
    # Filtrar exercício atual — usar str.contains por causa do encoding quebrado
    df = df[df['ORDEM_EXERC'].str.contains('LTIMO') & 
            ~df['ORDEM_EXERC'].str.contains('PEN')]
    
    # Manter só versão mais recente
    df = df.sort_values('VERSAO', ascending=False).drop_duplicates(subset=['CD_CONTA'])
    
    # Mostrar contas relevantes para os KPIs financeiros
    contas_relevantes = ['3.01', '3.02', '3.03', '3.04.01', '3.04.02', 
                         '3.04.03', '3.04.05', '3.09.01', '3.11.01']
    resultado = df[df['CD_CONTA'].isin(contas_relevantes)][
        ['CD_CONTA', 'DS_CONTA', 'VL_CONTA']
    ].sort_values('CD_CONTA')
    
    print(resultado.to_string(index=False))


=== BRB BANCO DE BRASILIA S.A. — CONTAS DRE 2014 ===
CD_CONTA                                     DS_CONTA   VL_CONTA
    3.01       Receitas da IntermediaÃ§Ã£o Financeira 2181587.00
    3.02       Despesas da IntermediaÃ§Ã£o Financeira -808074.00
    3.03   Resultado Bruto IntermediaÃ§Ã£o Financeira 1373513.00
 3.04.01         Receitas de PrestaÃ§Ã£o de ServiÃ§os  484934.00
 3.04.02                          Despesas de Pessoal -724605.00
 3.04.03              Outras Despesas Administrativas -422544.00
 3.04.05                 Outras Receitas Operacionais  147700.00
 3.09.01 AtribuÃ­do a SÃ³cios da Empresa Controladora  197462.00

=== BANESTES S.A. - BCO EST ESPIRITO SANTO — CONTAS DRE 2014 ===
CD_CONTA                                     DS_CONTA    VL_CONTA
    3.01       Receitas da IntermediaÃ§Ã£o Financeira  1752613.00
    3.02       Despesas da IntermediaÃ§Ã£o Financeira -1069151.00
    3.03   Resultado Bruto IntermediaÃ§Ã£o Financeira   683462.00
 3.04.01         Receitas de Pr

In [22]:
import pandas as pd

# Carregar BPP bruto de 2014
bpp_2014 = pd.read_csv(
    "../Balanco_passivo_bruto/dfp_bpp_2014.csv",
    sep=";",
    encoding="latin1"
)

bancos = ['BRB BANCO DE BRASILIA S.A.', 'BANESTES S.A. - BCO EST ESPIRITO SANTO']

for banco in bancos:
    print(f"\n=== {banco} — BPP 2014 ===")
    df = bpp_2014[bpp_2014['DENOM_CIA'] == banco].copy()
    
    # Filtrar exercício atual com str.contains por causa do encoding
    df = df[df['ORDEM_EXERC'].str.contains('LTIMO') & 
            ~df['ORDEM_EXERC'].str.contains('PEN')]
    
    # Manter só versão mais recente
    df = df.sort_values('VERSAO', ascending=False).drop_duplicates(subset=['CD_CONTA'])
    
    # Verificar contas de Patrimônio Líquido
    contas_pl = ['2.07', '2.08']
    resultado = df[df['CD_CONTA'].isin(contas_pl)][
        ['CD_CONTA', 'DS_CONTA', 'VL_CONTA']
    ].sort_values('CD_CONTA')
    
    print(resultado.to_string(index=False))


=== BRB BANCO DE BRASILIA S.A. — BPP 2014 ===
CD_CONTA                                                      DS_CONTA   VL_CONTA
    2.07 Passivos sobre Ativos NÃ£o Correntes a Venda e Descontinuados       0.00
    2.08                              PatrimÃ´nio LÃ­quido Consolidado 1379670.00

=== BANESTES S.A. - BCO EST ESPIRITO SANTO — BPP 2014 ===
CD_CONTA                                                      DS_CONTA   VL_CONTA
    2.07 Passivos sobre Ativos NÃ£o Correntes a Venda e Descontinuados       0.00
    2.08                              PatrimÃ´nio LÃ­quido Consolidado 1075185.00
